# Auto-Encoding Variational Bayes — Reproduction (Kaggle)
**Paper**: Kingma & Welling, 2013  
**Reproduces**: Figure 2 (ELBO curves), Figure 4 (2D manifold), Figure 5 (samples)  
**Dataset**: MNIST (binarized), optionally Frey Face

## 1. Imports & Config

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.io import loadmat
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# --- Hyperparams (from paper) ---
BATCH_SIZE    = 500
L             = 1
LR            = 0.01
WEIGHT_DECAY  = 1e-4
HIDDEN_DIM    = 500
HIDDEN_DIM_FF = 200
EPOCHS        = 200
LATENT_DIMS   = [2, 5, 10, 20]

## 1b. Kaggle Config

In [ ]:
# --- Kaggle paths ---
WORK_DIR     = Path('/kaggle/working')
CKPT_DIR     = WORK_DIR / 'checkpoints'
FIGURES_DIR  = WORK_DIR / 'figures'
DATA_DIR     = WORK_DIR / 'data'

for d in [CKPT_DIR, FIGURES_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Set RETRAIN=False to load saved checkpoints instead of retraining
RETRAIN = False
RETRAIN_WS = False

print(f'Checkpoints : {CKPT_DIR}')
print(f'Figures     : {FIGURES_DIR}')
print(f'Retrain     : {RETRAIN}')

## 2. Data

In [ ]:
binarize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x > 0.5).float())
])

mnist_train_raw = torchvision.datasets.MNIST(DATA_DIR, train=True,  download=True, transform=binarize)
mnist_test_raw  = torchvision.datasets.MNIST(DATA_DIR, train=False, download=True, transform=binarize)

# Précharge tout en GPU — élimine le bottleneck CPU/DataLoader
def preload_to_device(dataset, device):
    xs = torch.stack([dataset[i][0] for i in range(len(dataset))]).view(len(dataset), -1).to(device)
    return DataLoader(TensorDataset(xs), batch_size=BATCH_SIZE, shuffle=True)

BATCH_SIZE = 500
mnist_train_loader = preload_to_device(mnist_train_raw, DEVICE)
mnist_test_loader  = preload_to_device(mnist_test_raw,  DEVICE)

INPUT_DIM_MNIST = 784
print(f'MNIST — train: {len(mnist_train_raw)}, test: {len(mnist_test_raw)}, device: {DEVICE}')

In [ ]:
FF_PATH = Path('/kaggle/input/datasets/leparouxaristide/freyface/frey_rawface.mat')

def _is_valid_mat(path: Path) -> bool:
    header = path.read_bytes()[:6]
    return header[:6] == b'MATLAB' or header[:4] == b'\x00\x00\x00\x00'

def _parse_mat(path: Path) -> np.ndarray:
    """Try scipy first, fall back to h5py for v7.3 MAT files."""
    try:
        return loadmat(str(path))['ff'].T.astype(np.float32) / 255.0
    except Exception:
        import h5py
        with h5py.File(str(path), 'r') as f:
            return f['ff'][()].T.astype(np.float32) / 255.0

def load_frey_face(path: Path) -> tuple:
    if not path.exists():
        raise FileNotFoundError(f'{path} not found.')
    if not _is_valid_mat(path):
        raise ValueError(f'{path} is not a valid MAT file (bad magic bytes).')
    data  = _parse_mat(path)
    split = 1500
    train = TensorDataset(torch.tensor(data[:split]))
    test  = TensorDataset(torch.tensor(data[split:]))
    return train, test

try:
    ff_train, ff_test = load_frey_face(FF_PATH)
    ff_train_loader = DataLoader(ff_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    ff_test_loader  = DataLoader(ff_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    INPUT_DIM_FF = 560
    USE_FREY = True
    print(f'Frey Face — train: {len(ff_train)}, test: {len(ff_test)}')
except Exception as e:
    USE_FREY = False
    print(f'Frey Face not available: {e}. Skipping.')

## 3. Model

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.fc1       = nn.Linear(input_dim,  hidden_dim)
        self.fc_mu     = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = torch.tanh(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)


class BernoulliDecoder(nn.Module):
    def __init__(self, latent_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        h = torch.tanh(self.fc1(z))
        return torch.sigmoid(self.fc2(h))


class GaussianDecoder(nn.Module):
    def __init__(self, latent_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.fc1       = nn.Linear(latent_dim, hidden_dim)
        self.fc_mu     = nn.Linear(hidden_dim, output_dim)
        self.fc_logvar = nn.Linear(hidden_dim, output_dim)

    def forward(self, z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = torch.tanh(self.fc1(z))
        return torch.sigmoid(self.fc_mu(h)), self.fc_logvar(h)


class VAE(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int, decoder_type: str = 'bernoulli'):
        super().__init__()
        self.latent_dim   = latent_dim
        self.decoder_type = decoder_type
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        if decoder_type == 'bernoulli':
            self.decoder = BernoulliDecoder(latent_dim, hidden_dim, input_dim)
        else:
            self.decoder = GaussianDecoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

In [ ]:
class WakeSleepModel(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int, decoder_type: str = 'bernoulli'):
        super().__init__()
        self.latent_dim   = latent_dim
        self.decoder_type = decoder_type
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        if decoder_type == 'bernoulli':
            self.decoder = BernoulliDecoder(latent_dim, hidden_dim, input_dim)
        else:
            self.decoder = GaussianDecoder(latent_dim, hidden_dim, input_dim)

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)


## 4. ELBO Loss

In [ ]:
def elbo_bernoulli(x: torch.Tensor, x_recon: torch.Tensor,
                   mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    recon = F.binary_cross_entropy(x_recon, x, reduction='sum')
    kl    = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (recon + kl) / x.size(0)


def elbo_gaussian(x: torch.Tensor, mu_recon: torch.Tensor, logvar_recon: torch.Tensor,
                  mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    var_recon = logvar_recon.exp()
    recon = 0.5 * torch.sum(logvar_recon + (x - mu_recon).pow(2) / var_recon)
    kl    = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (recon + kl) / x.size(0)

## 5. Checkpoint helpers

In [ ]:
def ckpt_path(dataset: str, nz: int) -> Path:
    return CKPT_DIR / f'vae_{dataset}_nz{nz}.pt'


def save_checkpoint(path: Path, model: VAE, train_elbos: list, test_elbos: list) -> None:
    torch.save({
        'model_state':  model.state_dict(),
        'train_elbos':  train_elbos,
        'test_elbos':   test_elbos,
        'latent_dim':   model.latent_dim,
        'decoder_type': model.decoder_type,
    }, path)
    print(f'  Saved → {path.name}')


def load_checkpoint(path: Path, input_dim: int, hidden_dim: int) -> dict:
    ckpt  = torch.load(path, map_location=DEVICE, weights_only=False)
    model = VAE(input_dim, hidden_dim, ckpt['latent_dim'], ckpt['decoder_type']).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    print(f'  Loaded ← {path.name}')
    return {'model': model, 'train': ckpt['train_elbos'], 'test': ckpt['test_elbos']}

## 6. Training Loop

In [ ]:
def run_epoch(model: VAE, loader: DataLoader, optimizer, device: torch.device,
              train: bool = True) -> float:
    model.train() if train else model.eval()
    total = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.view(x.size(0), -1).to(device)
            out = model(x)
            if model.decoder_type == 'bernoulli':
                x_recon, mu, logvar = out
                loss = elbo_bernoulli(x, x_recon, mu, logvar)
            else:
                (mu_recon, logvar_recon), mu, logvar = out
                loss = elbo_gaussian(x, mu_recon, logvar_recon, mu, logvar)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item()
    return total / len(loader)


def train_vae(input_dim: int, hidden_dim: int, latent_dim: int,
              train_loader: DataLoader, test_loader: DataLoader,
              epochs: int = EPOCHS, decoder_type: str = 'bernoulli') -> dict:
    model = VAE(input_dim, hidden_dim, latent_dim, decoder_type).to(DEVICE)
    optimizer = torch.optim.Adagrad(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    train_elbos, test_elbos = [], []
    for epoch in range(epochs):
        tr = run_epoch(model, train_loader, optimizer, DEVICE, train=True)
        te = run_epoch(model, test_loader,  optimizer, DEVICE, train=False)
        train_elbos.append(-tr)
        test_elbos.append(-te)
        if (epoch + 1) % 20 == 0:
            print(f'  [{epoch+1:3d}/{epochs}] train ELBO={-tr:.2f}  test ELBO={-te:.2f}')
    return {'model': model, 'train': train_elbos, 'test': test_elbos}

In [ ]:
def wake_phase(model, x, optimizer_decoder):
    model.encoder.eval()
    model.decoder.train()
    
    mu, logvar = model.encode(x)
    z = model.reparameterize(mu, logvar).detach()
    
    if model.decoder_type == 'bernoulli':
        x_recon = model.decode(z)
        loss = F.binary_cross_entropy(x_recon, x, reduction='sum') / x.size(0)
    else:
        mu_r, logvar_r = model.decode(z)
        var_r = logvar_r.exp()
        loss = 0.5 * torch.sum(logvar_r + (x - mu_r).pow(2) / var_r) / x.size(0)
    
    optimizer_decoder.zero_grad()
    loss.backward()
    optimizer_decoder.step()
    return loss.item()

def sleep_phase(model, batch_size, optimizer_encoder):
    model.encoder.train()
    model.decoder.eval()
    
    z_prior = torch.randn(batch_size, model.latent_dim).to(DEVICE)
    
    with torch.no_grad():
        if model.decoder_type == 'bernoulli':
            x_fake = model.decode(z_prior)
            x_fake = (x_fake > 0.5).float()
        else:
            mu_fake, logvar_fake = model.decode(z_prior)
            x_fake = mu_fake
    
    x_fake = x_fake.detach()
    
    mu_enc, logvar_enc = model.encode(x_fake)
    log_q = -0.5 * torch.sum(
        logvar_enc + (z_prior - mu_enc).pow(2) / logvar_enc.exp() + np.log(2 * np.pi)
    ) / batch_size
    
    loss = -log_q
    optimizer_encoder.zero_grad()
    loss.backward()
    optimizer_encoder.step()
    return loss.item()

def eval_elbo_ws(model, loader):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for batch in loader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.view(x.size(0), -1).to(DEVICE)
            mu, logvar = model.encode(x)
            z = model.reparameterize(mu, logvar)
            
            if model.decoder_type == 'bernoulli':
                x_recon = model.decode(z)
                recon = F.binary_cross_entropy(x_recon, x, reduction='sum')
            else:
                mu_r, logvar_r = model.decode(z)
                var_r = logvar_r.exp()
                recon = 0.5 * torch.sum(logvar_r + (x - mu_r).pow(2) / var_r)
            
            kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            total += (recon + kl).item() / x.size(0)
    return -total / len(loader)


In [ ]:
def train_wake_sleep(input_dim, hidden_dim, latent_dim, train_loader, test_loader,
                     epochs=EPOCHS, decoder_type='bernoulli'):
    model = WakeSleepModel(input_dim, hidden_dim, latent_dim, decoder_type).to(DEVICE)
    
    opt_encoder = torch.optim.Adagrad(model.encoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    opt_decoder = torch.optim.Adagrad(model.decoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    
    train_elbos, test_elbos = [], []
    
    for epoch in range(epochs):
        if epoch == 0: print('  Démarrage de l\'epoch 1...')
        model.train()
        for batch in train_loader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.view(x.size(0), -1).to(DEVICE)
            wake_phase(model, x, opt_decoder)
            sleep_phase(model, x.size(0), opt_encoder)
        
        tr = eval_elbo_ws(model, train_loader)
        te = eval_elbo_ws(model, test_loader)
        train_elbos.append(tr)
        test_elbos.append(te)
        
        if (epoch + 1) % 1 == 0:
            print(f'  [{epoch+1:3d}/{epochs}] train={tr:.1f}  test={te:.1f}')
    
    return {'model': model, 'train': train_elbos, 'test': test_elbos}

def ckpt_path_ws(dataset, nz):
    return CKPT_DIR / f'ws_{dataset}_nz{nz}.pt'

def save_checkpoint_ws(path, model, train_elbos, test_elbos):
    torch.save({
        'model_state': model.state_dict(),
        'train_elbos': train_elbos,
        'test_elbos':  test_elbos,
        'latent_dim':  model.latent_dim,
        'decoder_type': model.decoder_type
    }, path)
    print(f'  Saved → {path.name}')

def load_checkpoint_ws(path, input_dim, hidden_dim):
    ckpt  = torch.load(path, map_location=DEVICE, weights_only=False)
    model = WakeSleepModel(input_dim, hidden_dim, ckpt['latent_dim'], ckpt['decoder_type']).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    print(f'  Loaded ← {path.name}')
    return {'model': model, 'train': ckpt['train_elbos'], 'test': ckpt['test_elbos']}


## 7. Experiment 1 — Train MNIST for all Nz (Figure 2)

In [ ]:
results_mnist = {}

for nz in LATENT_DIMS:
    path = ckpt_path('mnist', nz)
    if not RETRAIN and path.exists():
        print(f'\n=== MNIST, Nz={nz} — loading checkpoint ===')
        results_mnist[nz] = load_checkpoint(path, INPUT_DIM_MNIST, HIDDEN_DIM)
    else:
        print(f'\n=== MNIST, Nz={nz} — training ===')
        res = train_vae(INPUT_DIM_MNIST, HIDDEN_DIM, nz, mnist_train_loader, mnist_test_loader)
        save_checkpoint(path, res['model'], res['train'], res['test'])
        results_mnist[nz] = res

In [ ]:
results_ff = {}

if USE_FREY:
    for nz in LATENT_DIMS:
        path = ckpt_path('freyface', nz)
        if not RETRAIN and path.exists():
            print(f'\n=== Frey Face, Nz={nz} — loading checkpoint ===')
            results_ff[nz] = load_checkpoint(path, INPUT_DIM_FF, HIDDEN_DIM_FF)
        else:
            print(f'\n=== Frey Face, Nz={nz} — training ===')
            res = train_vae(INPUT_DIM_FF, HIDDEN_DIM_FF, nz,
                            ff_train_loader, ff_test_loader, decoder_type='gaussian')
            save_checkpoint(path, res['model'], res['train'], res['test'])
            results_ff[nz] = res

In [ ]:
results_ws_mnist = {}
for nz in LATENT_DIMS:
    path = ckpt_path_ws('mnist', nz)
    if not RETRAIN_WS and path.exists():
        print(f'\n=== WS MNIST Nz={nz} — chargement ===')
        results_ws_mnist[nz] = load_checkpoint_ws(path, INPUT_DIM_MNIST, HIDDEN_DIM)
    else:
        print(f'\n=== WS MNIST Nz={nz} — entraînement ===')
        res = train_wake_sleep(INPUT_DIM_MNIST, HIDDEN_DIM, nz, mnist_train_loader, mnist_test_loader)
        save_checkpoint_ws(path, res['model'], res['train'], res['test'])
        results_ws_mnist[nz] = res


In [ ]:
results_ws_ff = {}
if USE_FREY:
    for nz in LATENT_DIMS:
        path = ckpt_path_ws('freyface', nz)
        if not RETRAIN_WS and path.exists():
            print(f'\n=== WS Frey Face Nz={nz} — chargement ===')
            results_ws_ff[nz] = load_checkpoint_ws(path, INPUT_DIM_FF, HIDDEN_DIM_FF)
        else:
            print(f'\n=== WS Frey Face Nz={nz} — entraînement ===')
            res = train_wake_sleep(INPUT_DIM_FF, HIDDEN_DIM_FF, nz,
                                   ff_train_loader, ff_test_loader, decoder_type='gaussian')
            save_checkpoint_ws(path, res['model'], res['train'], res['test'])
            results_ws_ff[nz] = res


## 12. Final ELBO Summary

In [ ]:
print('=== Final ELBO (test) per Nz ===')
for nz in LATENT_DIMS:
    final_elbo = results_mnist[nz]['test'][-1]
    print(f'  Nz={nz:3d}  ELBO = {final_elbo:.2f}')

# Paper reports approx -100 to -120 nats on MNIST test set for AEVB
print('\n=== Final ELBO (test) WS ===')
for nz in sorted(results_ws_mnist):
    print(f'  WS MNIST    Nz={nz:2d}  ELBO = {results_ws_mnist[nz]["test"][-1]:.2f}')
for nz in sorted(results_ws_ff):
    print(f'  WS FreyFace Nz={nz:2d}  ELBO = {results_ws_ff[nz]["test"][-1]:.2f}')


In [ ]:
print('=== Checkpoints sauvegardés ===')
for f in sorted(CKPT_DIR.glob('*.pt')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name}  ({size_mb:.1f} MB)')

print('\n=== Figures sauvegardées ===')
for f in sorted(FIGURES_DIR.glob('*.png')):
    size_kb = f.stat().st_size / 1e3
    print(f'  {f.name}  ({size_kb:.0f} KB)')

print('\n⚠️  IMPORTANT : clique sur "Save & Run All" (bouton en haut à droite)')
print('   pour committer les outputs — sinon /kaggle/working est perdu à la fin de la session.')

## 13. Package MNIST Checkpoints for Download

In [ ]:
import zipfile

zip_path = WORK_DIR / 'mnist_checkpoints.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for nz in LATENT_DIMS:
        p = ckpt_path('mnist', nz)
        if p.exists():
            zf.write(p, p.name)
            print(f'  Added {p.name}')
        else:
            print(f'  [!] Manquant : {p.name}')

print(f'\nZip prêt : {zip_path}')
print('Télécharge via : Kaggle → Output → mnist_checkpoints.zip')
    # Add WS checkpoints as well
    for nz in LATENT_DIMS:
        p = ckpt_path_ws('mnist', nz)
        if p.exists():
            zf.write(p, p.name)
            print(f'  Added {p.name}')
    for nz in LATENT_DIMS:
        p = ckpt_path_ws('freyface', nz)
        if p.exists():
            zf.write(p, p.name)
            print(f'  Added {p.name}')
